In [1]:
import xarray as xr
from dask.distributed import Client

In [2]:
chunk_sizes = {'time_counter': 1, 'deptht': 40, 'gridY': 898, 'gridX': 398}

In [3]:
ww_out_yr = xr.open_dataset('/ocean/cdonaldson/scrubbers/calcs/SalishSea_1d_20220101_20221231_scrb_T.nc', chunks=chunk_sizes)

/home/cdonaldson/conda_envs/analysis-2/lib/python3.12/site-packages/xarray/core/dataset.py:278: UserWarning: The specified chunks separate the stored chunks along dimension "time_counter" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(


In [4]:
client=Client(processes=True, n_workers=4, local_directory='/tmp/cdonaldson')  # 4 is a good choice for n_workers even on salish, if I was the only person logged on or asked for space, could go as high as 16 but... be conscious of this
# threads_per_worker = 1 if needed, netcdf is not threadsafe now so be realiable by specifying 1 (at expense of just some efficiency)

In [5]:
low_seas1 = slice(0,130)
low_seas2 = slice(276, -1)
high_seas = slice(131, 275)

ww_ls1 = ww_out_yr.isel(time_counter=low_seas1)
ww_ls2 = ww_out_yr.isel(time_counter=low_seas2)
ww_ls = xr.concat([ww_ls1, ww_ls2], dim='time_counter')

ww_hs = ww_out_yr.isel(time_counter=high_seas)

In [6]:
ww_ls_mean = ww_ls.mean(dim='time_counter')
ww_ls_mean = ww_ls_mean.assign_coords(season=('season', ['LOW']))

In [7]:
ww_hs_mean = ww_hs.mean(dim='time_counter')
ww_hs_mean = ww_hs_mean.assign_coords(season=('season', ['HIGH']))

In [8]:
ww_mon_mean =  ww_out_yr.groupby('time_counter.month').mean()

In [9]:
ww_cal_mean = ww_out_yr.groupby('time_counter.season').mean()

In [10]:
ww_seas_mean = xr.concat([ww_cal_mean, ww_ls_mean, ww_hs_mean], dim='season')

In [11]:
ww_seas_mean.to_netcdf('/ocean/cdonaldson/scrubbers/calcs/SalishSea_1d_20220101_20221231_scrb_T_seas.nc', engine='netcdf4')
ww_mon_mean.to_netcdf('/ocean/cdonaldson/scrubbers/calcs/SalishSea_1d_20220101_20221231_scrb_T_mon.nc', engine='netcdf4')

In [12]:
client.close()